# Demo: Branching MASCAF

Fit a cable morphology to the branching demo mesh. The skeleton stops short of the branch ends, so the fit is extended at each tip and then scaled to the mesh surface area.

In [1]:
from mascaf import *

import logging

logging.basicConfig(level=logging.INFO)

## Mesh and skeleton

Load the demo mesh and its curve skeleton. The skeleton is the centerline the cable fit follows.

In [2]:
mm = MeshManager(mesh_path="../../data/demo/branching.obj")
mm.print_mesh_analysis()
raw_skeleton = SkeletonGraph.from_txt(f"../../data/demo/branching.polylines.txt")
# raw_skeleton.prune_short_branches_inplace(min_length_fraction=1)
mesh_fig, camera = visualize_mesh_3d(
    mesh=mm,
    title="Branching test model",
    skel=raw_skeleton,
    show_axes=False,
    return_camera=True,
)
mesh_fig

INFO:mascaf.mesh:Loaded mesh: 1417 vertices, 2830 faces


INFO:mascaf.mesh:Mesh Analysis Report
INFO:mascaf.mesh:====================
INFO:mascaf.mesh:
Geometry:
INFO:mascaf.mesh:  * Vertices: 1417
INFO:mascaf.mesh:  * Faces: 2830
INFO:mascaf.mesh:  * Components: 1
INFO:mascaf.mesh:  * Volume: 5.11
INFO:mascaf.mesh:  * Bounds: [-1.0, -1.3, -3.6] to [1.0, 1.4, 1.0]
INFO:mascaf.mesh:
Mesh Quality:
INFO:mascaf.mesh:  * Watertight: True
INFO:mascaf.mesh:  * Winding Consistent: True
INFO:mascaf.mesh:  * Normal Direction: outward
INFO:mascaf.mesh:  * Duplicate Vertices: 0
INFO:mascaf.mesh:  * Degenerate Faces: 0
INFO:mascaf.mesh:
Topology:
INFO:mascaf.mesh:  * Genus: 0
INFO:mascaf.mesh:  * Euler Characteristic: 2
INFO:mascaf.mesh:
No issues detected
INFO:mascaf.mesh:
Recommendation:
INFO:mascaf.mesh:  Mesh appears to be in good condition.
INFO:mascaf.mesh:====================


## Cable fit

Basis optimization snaps nodes onto the mesh and pulls them toward the centerline. `CableFitter` then resamples that skeleton so edges are at most `max_edge_length` and estimates a radius at each sample from the mesh cross-section.

In [3]:
optimizer_options = BasisOptimizerOptions(
    do_pruning=False,
    do_snapping=True,
    do_forcing=True,
    max_iterations=5,
    alpha_s=0.1,
    preserve_terminal_nodes=False,
    preserve_branch_nodes=False,
)
skeleton = raw_skeleton

In [4]:
max_edge_length = 0.7

swc_filepath = "../../data/demo/branching.swc"

morph = CableFitter(
    FitOptions(
        max_edge_length=max_edge_length,
        basis_optimizer_options=optimizer_options,
    )
).fit(
    mm.mesh,
    skeleton,
)

INFO:mascaf.cable_fitting:Starting cable fit with 137 skeleton nodes, 136 skeleton edges, 1417 mesh vertices, and max_edge_length=0.7
INFO:mascaf.morphology_graph:Resampled morphology: 13 nodes, 12 edges (max_edge_length=0.7000, source 137 nodes / 136 edges, 5 sections)
INFO:mascaf.cable_fitting:Optimizing morphology basis before radius fitting
INFO:mascaf.basis_optimizer:Starting basis optimization...
INFO:mascaf.basis_optimizer:  Input: 13 nodes, 12 edges, total length 5.8190
INFO:mascaf.basis_optimizer:Phase 1 - Snapping: 0 nodes outside mesh
INFO:mascaf.basis_optimizer:Phase 2 - Forcing: max 5 iterations
INFO:mascaf.basis_optimizer:  Forcing alpha_s=0.1000 step_scale=0.5000 ray_jitter=0.0000 active_resample=False
INFO:mascaf.basis_optimizer:  Iteration 0: avg movement = 0.032797, centering_error = 9.332129e-01
INFO:mascaf.basis_optimizer:  Iteration 1: avg movement = 0.024541, centering_error = 7.107037e-01
INFO:mascaf.basis_optimizer:  Iteration 2: avg movement = 0.019492, centeri

## Fitted cable

The mesh and skeleton are on the left; the cable model is on the right. Both panels use the same view. Validation compares cable volume and surface area with the mesh.

In [5]:
fig = visualize_mesh_cable_3d(mm, skeleton, morph, show_axes=False, camera=camera)
fig.show()

validator = Validation(mm, skeleton, morph)
validator.full_validation()

INFO:swctools.geometry:batch_frusta count=12 sides=16 end_caps=False verts=384 faces=384
INFO:swctools.geometry:FrustaSet.from_swc_model edges=12 sides=16 end_caps=False


INFO:swctools.viz:plot_model slider=False frusta=12 show_frusta=True show_centroid=True


INFO:mascaf.validation:Initialized Validation from MorphologyGraph
INFO:mascaf.validation:  Mesh: 1417 vertices, 2830 faces
INFO:mascaf.validation:  Skeleton: 137 nodes, 136 edges
INFO:mascaf.validation:  MorphologyGraph: 13 nodes, 12 edges
INFO:mascaf.validation:Validation Results, account_for_overlaps=False:
INFO:mascaf.validation:-- Volume Comparison:
INFO:mascaf.validation:---- Mesh volume:       5.1071
INFO:mascaf.validation:---- Morphology volume: 3.0690
INFO:mascaf.validation:---- Ratio:             0.6009
INFO:mascaf.validation:---- Error:             -2.0382
INFO:mascaf.validation:---- Relative error:    -39.91%
INFO:mascaf.validation:-- Surface Area Comparison:
INFO:mascaf.validation:---- Mesh area:         22.3177
INFO:mascaf.validation:---- Morphology area:   16.9427
INFO:mascaf.validation:---- Ratio:             0.7592
INFO:mascaf.validation:---- Error:             -5.3750
INFO:mascaf.validation:---- Relative error:    -24.08%
INFO:mascaf.validation:Validation Results, acc

## Terminal extension

Each branch tip is continued along its outward tangent by one parent radius. The new tip radius is half the parent radius, so the end cap sits closer to the mesh and covers area the short skeleton missed.

In [6]:
morph.extend_terminals()
fig = visualize_mesh_cable_3d(mm, skeleton, morph, show_axes=False, camera=camera)
fig.show()

validator = Validation(mm, skeleton, morph)
validator.full_validation()

INFO:swctools.geometry:batch_frusta count=16 sides=16 end_caps=False verts=512 faces=512
INFO:swctools.geometry:FrustaSet.from_swc_model edges=16 sides=16 end_caps=False
INFO:swctools.viz:plot_model slider=False frusta=16 show_frusta=True show_centroid=True


INFO:mascaf.validation:Initialized Validation from MorphologyGraph
INFO:mascaf.validation:  Mesh: 1417 vertices, 2830 faces
INFO:mascaf.validation:  Skeleton: 137 nodes, 136 edges
INFO:mascaf.validation:  MorphologyGraph: 17 nodes, 16 edges
INFO:mascaf.validation:Validation Results, account_for_overlaps=False:
INFO:mascaf.validation:-- Volume Comparison:
INFO:mascaf.validation:---- Mesh volume:       5.1071
INFO:mascaf.validation:---- Morphology volume: 5.5806
INFO:mascaf.validation:---- Ratio:             1.0927
INFO:mascaf.validation:---- Error:             0.4735
INFO:mascaf.validation:---- Relative error:    9.27%
INFO:mascaf.validation:-- Surface Area Comparison:
INFO:mascaf.validation:---- Mesh area:         22.3177
INFO:mascaf.validation:---- Morphology area:   22.9105
INFO:mascaf.validation:---- Ratio:             1.0266
INFO:mascaf.validation:---- Error:             0.5928
INFO:mascaf.validation:---- Relative error:    2.66%
INFO:mascaf.validation:Validation Results, account_f

## Surface-area normalization

Scale every radius by one factor so the cable surface area matches the mesh. Overlap between branches is left uncorrected.

In [7]:
morph.scale_radii_to_match_mesh(
    mm.mesh, metric="surface_area", account_for_overlaps=False
)
fig = visualize_mesh_cable_3d(mm, skeleton, morph, show_axes=False, camera=camera)
fig.show()


validator = Validation(mm, skeleton, morph)
validator.full_validation()

INFO:swctools.geometry:batch_frusta count=16 sides=16 end_caps=False verts=512 faces=512
INFO:swctools.geometry:FrustaSet.from_swc_model edges=16 sides=16 end_caps=False


INFO:swctools.viz:plot_model slider=False frusta=16 show_frusta=True show_centroid=True


INFO:mascaf.validation:Initialized Validation from MorphologyGraph
INFO:mascaf.validation:  Mesh: 1417 vertices, 2830 faces
INFO:mascaf.validation:  Skeleton: 137 nodes, 136 edges
INFO:mascaf.validation:  MorphologyGraph: 17 nodes, 16 edges
INFO:mascaf.validation:Validation Results, account_for_overlaps=False:
INFO:mascaf.validation:-- Volume Comparison:
INFO:mascaf.validation:---- Mesh volume:       5.1071
INFO:mascaf.validation:---- Morphology volume: 5.3413
INFO:mascaf.validation:---- Ratio:             1.0458
INFO:mascaf.validation:---- Error:             0.2341
INFO:mascaf.validation:---- Relative error:    4.58%
INFO:mascaf.validation:-- Surface Area Comparison:
INFO:mascaf.validation:---- Mesh area:         22.3177
INFO:mascaf.validation:---- Morphology area:   22.3177
INFO:mascaf.validation:---- Ratio:             1.0000
INFO:mascaf.validation:---- Error:             0.0000
INFO:mascaf.validation:---- Relative error:    0.00%
INFO:mascaf.validation:Validation Results, account_f